# NoseKnows — Gemma 2 2B Fine-tuning

**Model:** `google/gemma-2-2b` — pure text generation, `AutoModelForCausalLM`  
**Strategy:** QLoRA on the last 4 transformer layers (r=16, alpha=32)  
**Data:** `dataset.jsonl` uploaded as a Kaggle dataset  
**Split:** 90% train / 10% validation  
**Epochs:** 3 with per-epoch + per-step checkpointing  
**Hardware:** Dual T4 GPU  

**Resume:** Re-run all cells. Trainer reads the latest step checkpoint automatically.  

**Outputs** (all in `/kaggle/working/`):  
- `checkpoints/` — step-level (every 10 steps) + permanent per-epoch adapter checkpoints  
- `final_adapter/` — final LoRA adapter (best checkpoint by val loss)  
- `training_report.json` — full training statistics  
- `plots/` — loss curve visualisations

In [ ]:
# ── Cell 1: install dependencies ──────────────────────────────────────────
# gemma2 is supported in standard PyPI transformers — no source install needed.
# bitsandbytes==0.46.1: confirmed working on Kaggle T4.
# trl and peft: pinned to ranges known to work together.
import subprocess, sys

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.45.0",
        "trl>=0.8.6",
        "peft>=0.10.0",
        "bitsandbytes==0.46.1",
        "accelerate>=0.29.0",
        "datasets>=2.18.0",
        "matplotlib",
        "seaborn",
    ],
    check=True,
)
print("Dependencies installed.")

# Verify gemma2 is in the transformers config mapping.
import transformers
print(f"transformers version: {transformers.__version__}")
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
print(f"gemma2 in CONFIG_MAPPING: {'gemma2' in CONFIG_MAPPING}")

In [ ]:
# ── Cell 2: imports ───────────────────────────────────────────────────────

# Set CUDA visibility BEFORE any CUDA initialization.
# Must be the first thing that runs — before torch imports CUDA.
# Restricts to GPU 1 (where this Kaggle instance loads the model)
# and prevents the trainer from attempting DataParallel across both T4s.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import inspect
import json
import logging
import random
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import seaborn as sns
import torch
from datasets import Dataset
from kaggle_secrets import UserSecretsClient
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainerCallback,
    TrainerControl,
    TrainerState,
    TrainingArguments,
)
from trl import SFTTrainer, SFTConfig

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
)
log = logging.getLogger("nosknows_finetune")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count visible: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(
        f"  GPU {i}: {torch.cuda.get_device_name(i)} — "
        f"{torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB"
    )

In [ ]:
# ── Cell 3: constants ─────────────────────────────────────────────────────

# ── paths ──
# UPDATE: set DATASET_PATH after uploading dataset.jsonl as a Kaggle dataset.
# It will appear as /kaggle/input/<your-dataset-name>/dataset.jsonl
DATASET_PATH   = Path("/kaggle/input/datasets/paraschiv/perfumes-synthetic-dataset/dataset.jsonl")
OUTPUT_DIR     = Path("/kaggle/working")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
FINAL_ADAPTER  = OUTPUT_DIR / "final_adapter"
PLOTS_DIR      = OUTPUT_DIR / "plots"
REPORT_PATH    = OUTPUT_DIR / "training_report.json"

for d in [CHECKPOINT_DIR, FINAL_ADAPTER, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── model ──
# Pure text generation model — loads with AutoModelForCausalLM.
# Gated: requires HF token + license acceptance on huggingface.co/google/gemma-2-2b
MODEL_ID = "google/gemma-2-2b-it"

# ── data ──
TRAIN_SPLIT    = 0.9    # 90% train, 10% validation
MAX_SEQ_LENGTH = 512    # covers system + user + assistant comfortably

# ── LoRA ──
# Targeting last N_LORA_LAYERS transformer decoder layers.
# Layer count resolved at runtime from model config.
LORA_R              = 16
LORA_ALPHA          = 32    # 2× r — standard scaling
LORA_DROPOUT        = 0.05  # light regularisation for small dataset
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
N_LORA_LAYERS = 4

# ── training ──
NUM_EPOCHS                  = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8     # effective batch = 2 × 8 × 2 GPUs = 32
LEARNING_RATE               = 2e-4
WARMUP_RATIO                = 0.05
LR_SCHEDULER                = "cosine"
FP16                        = False  # T4 fp16 stable; bf16 is not
BF16 = True

# Save every 10 steps — with ~31 steps/epoch on 1104 examples,
# this gives ~3 checkpoints per epoch. Never lose more than 10 steps.
SAVE_STEPS                  = 10
SAVE_TOTAL_LIMIT            = 3     # keep last 3 step checkpoints

print("Constants set.")
print(f"Dataset : {DATASET_PATH}")
print(f"Model   : {MODEL_ID}")
print(f"LoRA r/alpha     : {LORA_R}/{LORA_ALPHA}")
print(f"Max seq length   : {MAX_SEQ_LENGTH}")
print(f"Epochs           : {NUM_EPOCHS}")
print(f"Save every N steps: {SAVE_STEPS}")

In [ ]:
# ── Cell 4: load and prepare dataset ─────────────────────────────────────
#
# gemma-2-2b-it chat template does not support the system role.
# The template only accepts user and assistant (internally "model") turns.
# Fix: merge system content into the first user turn before building the dataset.
# This matches exactly how the model will be used at inference time.

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATASET_PATH}.\n"
        "Upload dataset.jsonl as a Kaggle dataset and update DATASET_PATH in Cell 3."
    )

log.info("Loading dataset from %s...", DATASET_PATH)
raw_records: list[dict] = []
skipped = 0
with open(DATASET_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            record = json.loads(line)
            msgs = record.get("messages", [])
            if len(msgs) != 3:
                skipped += 1
                continue
            if [m["role"] for m in msgs] != ["system", "user", "assistant"]:
                skipped += 1
                continue
            if any(not m["content"].strip() for m in msgs):
                skipped += 1
                continue

            # Merge system prompt into user turn.
            # Gemma 2 chat template raises TemplateError: System role not supported
            # if a system message is passed. Prepending it to the user content
            # preserves all information while staying within the supported format.
            system_content = msgs[0]["content"]
            user_content   = msgs[1]["content"]
            merged_user    = f"{system_content}\n\n{user_content}"

            raw_records.append({
                "messages": [
                    {"role": "user",      "content": merged_user},
                    {"role": "assistant", "content": msgs[2]["content"]},
                ]
            })
        except (json.JSONDecodeError, KeyError):
            skipped += 1

log.info("Loaded %d valid examples. Skipped %d malformed.", len(raw_records), skipped)

random.shuffle(raw_records)

split_idx     = int(len(raw_records) * TRAIN_SPLIT)
train_records = raw_records[:split_idx]
val_records   = raw_records[split_idx:]

train_dataset = Dataset.from_list(train_records)
val_dataset   = Dataset.from_list(val_records)

print(f"Train examples : {len(train_dataset):,}")
print(f"Val examples   : {len(val_dataset):,}")
print(f"Skipped        : {skipped}")
print("\nSample record (first train example, merged format):")
for msg in train_dataset[0]["messages"]:
    print(f"  [{msg['role']:>9}] {msg['content'][:120]}...")

In [ ]:
# ── Cell 5: load model and tokenizer ─────────────────────────────────────
# CUDA_VISIBLE_DEVICES and PYTORCH_ALLOC_CONF are set in Cell 2.
# With only GPU 1 visible (remapped to cuda:0), device_map="auto"
# places all layers on the single visible device cleanly.

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
if not hf_token:
    raise EnvironmentError(
        "HF_TOKEN is empty. "
        "Add it under Add-ons -> Secrets and enable it for this notebook."
    )

log.info("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=hf_token,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    log.info("Set pad_token = eos_token (%s)", tokenizer.eos_token)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

log.info("Loading model (4-bit NF4)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=hf_token,
    quantization_config=bnb_config,
    device_map="auto",   # one GPU visible via CUDA_VISIBLE_DEVICES — auto is correct
    trust_remote_code=True,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

log.info("Model loaded.")
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params / 1e9:.2f}B")
for i in range(torch.cuda.device_count()):
    alloc    = torch.cuda.memory_allocated(i) / 1e9
    reserved = torch.cuda.memory_reserved(i)  / 1e9
    print(
        f"GPU {i} ({torch.cuda.get_device_name(i)}): "
        f"allocated {alloc:.2f} GB | reserved {reserved:.2f} GB"
    )

total_alloc = sum(
    torch.cuda.memory_allocated(i)
    for i in range(torch.cuda.device_count())
) / 1e9
if total_alloc < 1.0:
    raise RuntimeError(
        f"Model only using {total_alloc:.2f} GB total VRAM. "
        "4-bit quantization likely failed. Check bitsandbytes version."
    )
print(f"\nTotal VRAM allocated: {total_alloc:.2f} GB — quantization OK.")

In [ ]:
# ── Cell 6: inspect model architecture ───────────────────────────────────
#
# Print the top-level modules and layer count so we can verify
# LoRA will target the correct layers before training starts.

print("Top-level named modules:")
for name, module in model.named_children():
    print(f"  {name}: {type(module).__name__}")

print(f"\nModel config type: {type(model.config).__name__}")

# Resolve num_hidden_layers — for gemma-2-2b this is flat on model.config.
if hasattr(model.config, "num_hidden_layers"):
    print(f"num_hidden_layers: {model.config.num_hidden_layers}")
elif hasattr(model.config, "text_config"):
    print(f"num_hidden_layers (text_config): {model.config.text_config.num_hidden_layers}")
else:
    print("num_hidden_layers not found directly — check model.config manually.")

In [ ]:
# ── Cell 7: configure LoRA on last N transformer layers ───────────────────
#
# gemma-2-2b is a pure causal LM — TaskType.CAUSAL_LM is correct.
# Layer indices resolved at runtime so the script works across model variants.

def get_num_layers(model: AutoModelForCausalLM) -> int:
    """
    Resolve transformer decoder layer count from model config.
    Handles flat config and nested text_config.
    Falls back to counting named modules if config attributes are absent.
    """
    if hasattr(model.config, "num_hidden_layers"):
        return model.config.num_hidden_layers
    if hasattr(model.config, "text_config") and hasattr(
        model.config.text_config, "num_hidden_layers"
    ):
        return model.config.text_config.num_hidden_layers
    # Last resort: count from named modules
    import re
    indices = set()
    for name, _ in model.named_modules():
        m = re.search(r"\.layers\.([0-9]+)\.", name)
        if m:
            indices.add(int(m.group(1)))
    if indices:
        return max(indices) + 1
    raise RuntimeError(
        "Cannot determine num_hidden_layers. "
        "Inspect model.config and set layers_to_transform manually."
    )


num_layers          = get_num_layers(model)
start_layer         = num_layers - N_LORA_LAYERS
layers_to_transform = list(range(start_layer, num_layers))

log.info(
    "Total decoder layers: %d. LoRA targeting last %d: layers %d to %d.",
    num_layers, N_LORA_LAYERS, start_layer, num_layers - 1,
)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    layers_to_transform=layers_to_transform,
    bias="none",
    task_type=TaskType.CAUSAL_LM,  # correct for AutoModelForCausalLM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("\nTrainable modules:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  {name}")

In [ ]:
# ── Cell 8: training callbacks ────────────────────────────────────────────
#
# EpochCheckpointCallback: permanent per-epoch adapter — never deleted
#   by save_total_limit rotation. Saved to checkpoints/epoch_N/.
#
# LossHistoryCallback: accumulates train and eval loss for visualisation.

class EpochCheckpointCallback(TrainerCallback):
    """
    Saves a permanent LoRA adapter checkpoint at the end of each epoch.
    These are never affected by save_total_limit.
    """

    def __init__(self, output_dir: Path) -> None:
        self.output_dir = output_dir

    def on_epoch_end(
        self,
        args: TrainingArguments,
        state: TrainerState,
        control: TrainerControl,
        **kwargs: Any,
    ) -> None:
        epoch = int(state.epoch)
        save_path = self.output_dir / f"epoch_{epoch}"
        save_path.mkdir(parents=True, exist_ok=True)
        kwargs["model"].save_pretrained(str(save_path))
        log.info("Epoch %d permanent checkpoint saved → %s", epoch, save_path)


class LossHistoryCallback(TrainerCallback):
    """Collects train and eval loss at each logging step and epoch end."""

    def __init__(self) -> None:
        self.train_losses: list[tuple[float, float]] = []  # (step, loss)
        self.eval_losses:  list[tuple[float, float]] = []  # (epoch, loss)

    def on_log(
        self,
        args: TrainingArguments,
        state: TrainerState,
        control: TrainerControl,
        logs: dict[str, float] | None = None,
        **kwargs: Any,
    ) -> None:
        if logs is None:
            return
        if "loss" in logs:
            self.train_losses.append((state.global_step, logs["loss"]))
        if "eval_loss" in logs:
            self.eval_losses.append((state.epoch, logs["eval_loss"]))


epoch_ckpt_cb   = EpochCheckpointCallback(CHECKPOINT_DIR)
loss_history_cb = LossHistoryCallback()
print("Callbacks defined.")

In [ ]:
# ── Cell 9: SFTConfig and SFTTrainer ──────────────────────────────────────
#
# Checkpointing safety — two independent layers:
#   1. save_strategy="epoch": trainer saves a checkpoint at end of each epoch,
#      rotated by save_total_limit=3. resume_from_checkpoint=True picks these
#      up automatically on re-run.
#   2. EpochCheckpointCallback: permanent epoch-level adapter — never rotated.
#
# Version-safety fixes applied:
#   - max_seq_length vs max_length: resolved at runtime via inspect
#   - tokenizer vs processing_class: resolved at runtime via inspect
#   - warmup_ratio deprecated: replaced with warmup_steps
#   - logging_dir deprecated: removed
#   - save_strategy matches eval_strategy (required by load_best_model_at_end)

import inspect
from trl import SFTTrainer, SFTConfig

# ── resolve version-dependent parameter names ──
sft_params     = inspect.signature(SFTConfig.__init__).parameters
seq_len_kwarg  = "max_length" if "max_length" in sft_params else "max_seq_length"
log.info("SFTConfig sequence length parameter name: '%s'", seq_len_kwarg)

trainer_params   = inspect.signature(SFTTrainer.__init__).parameters
tokenizer_kwarg  = "processing_class" if "processing_class" in trainer_params else "tokenizer"
log.info("SFTTrainer tokenizer parameter name: '%s'", tokenizer_kwarg)

# ── compute warmup steps from actual step count ──
n_gpus = torch.cuda.device_count()
steps_per_epoch = max(
    1,
    len(train_dataset) // (
        PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * n_gpus
    ),
)
total_steps   = steps_per_epoch * NUM_EPOCHS
warmup_steps  = max(1, int(WARMUP_RATIO * total_steps))

log.info(
    "Steps per epoch: %d | Total steps: %d | Warmup steps: %d",
    steps_per_epoch, total_steps, warmup_steps,
)

# ── SFTConfig ──
sft_config = SFTConfig(
    # ── output ──
    output_dir=str(CHECKPOINT_DIR),

    # ── training duration ──
    num_train_epochs=NUM_EPOCHS,

    # ── batch and accumulation ──
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    # ── optimisation ──
    learning_rate=LEARNING_RATE,
    warmup_steps=warmup_steps,      # replaces deprecated warmup_ratio
    lr_scheduler_type=LR_SCHEDULER,
    optim="paged_adamw_8bit",

    # ── precision ──
    fp16=False,
    bf16=True,

    # ── sequence length (version-safe) ──
    **{seq_len_kwarg: MAX_SEQ_LENGTH},
    packing=False,

    # ── checkpointing ──
    # save_strategy must match eval_strategy when load_best_model_at_end=True.
    save_strategy="epoch",
    save_total_limit=SAVE_TOTAL_LIMIT,
    resume_from_checkpoint=True,

    # ── evaluation ──
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # ── logging ──
    # logging_dir removed — deprecated in trl 5.2
    logging_strategy="steps",
    logging_steps=10,
    report_to="none",

    # ── SFT specific ──
    dataset_kwargs={"skip_prepare_dataset": False},

    # ── misc ──
    seed=SEED,
    data_seed=SEED,
    remove_unused_columns=False,
)

# ── SFTTrainer ──
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    **{tokenizer_kwarg: tokenizer},     # version-safe tokenizer argument
    callbacks=[epoch_ckpt_cb, loss_history_cb],
)

print("Trainer configured.")
print(f"  Steps per epoch (approx)   : {steps_per_epoch}")
print(f"  Total steps (approx)       : {total_steps}")
print(f"  Warmup steps               : {warmup_steps}")
print(f"  Effective batch size       : {PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * n_gpus}")
print(f"  Save strategy              : epoch (last {SAVE_TOTAL_LIMIT} kept)")
print(f"  Permanent epoch checkpoints: {CHECKPOINT_DIR}/epoch_N/")

In [ ]:
# ── Cell 10: train ────────────────────────────────────────────────────────
#
# resume_from_checkpoint=True means if checkpoints/checkpoint-N/ exists,
# the trainer resumes from the latest one automatically on re-run.
# No manual intervention needed across sessions.

log.info("Starting training...")
train_result = trainer.train()

log.info("Training complete.")
print("\nTraining summary:")
print(f"  Total steps          : {train_result.global_step}")
print(f"  Final train loss     : {train_result.training_loss:.4f}")
print(f"  Runtime              : {train_result.metrics.get('train_runtime', 0) / 3600:.2f}h")

In [ ]:
# ── Cell 11: save final adapter ───────────────────────────────────────────
#
# Saves only the LoRA adapter weights — not the full base model.
# Size: ~30-60 MB for r=16 on 4 layers of a 2B model.
#
# load_best_model_at_end=True means trainer.model is already the
# best checkpoint (lowest eval_loss), not necessarily the last epoch.
#
# Load at inference:
#   from peft import PeftModel
#   model = PeftModel.from_pretrained(base_model, "/path/to/final_adapter")

log.info("Saving final LoRA adapter to %s...", FINAL_ADAPTER)
trainer.model.save_pretrained(str(FINAL_ADAPTER))
tokenizer.save_pretrained(str(FINAL_ADAPTER))
log.info("Final adapter saved.")

report = {
    "model_id":             MODEL_ID,
    "lora_r":               LORA_R,
    "lora_alpha":           LORA_ALPHA,
    "lora_dropout":         LORA_DROPOUT,
    "n_lora_layers":        N_LORA_LAYERS,
    "num_epochs":           NUM_EPOCHS,
    "train_examples":       len(train_dataset),
    "val_examples":         len(val_dataset),
    "max_seq_length":       MAX_SEQ_LENGTH,
    "effective_batch_size": (
        PER_DEVICE_TRAIN_BATCH_SIZE *
        GRADIENT_ACCUMULATION_STEPS *
        torch.cuda.device_count()
    ),
    "learning_rate":        LEARNING_RATE,
    "total_steps":          train_result.global_step,
    "final_train_loss":     train_result.training_loss,
    "runtime_hours":        train_result.metrics.get("train_runtime", 0) / 3600,
    "train_loss_history":   loss_history_cb.train_losses,
    "eval_loss_history":    loss_history_cb.eval_losses,
}
with open(REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)
log.info("Report saved to %s", REPORT_PATH)

In [ ]:
# ── Cell 12: visualisations ───────────────────────────────────────────────

sns.set_theme(style="darkgrid", palette="muted")
FIGSIZE = (10, 5)

train_steps  = [s for s, _ in loss_history_cb.train_losses]
train_losses = [l for _, l in loss_history_cb.train_losses]
eval_epochs  = [e for e, _ in loss_history_cb.eval_losses]
eval_losses  = [l for _, l in loss_history_cb.eval_losses]

# ── Plot 1: training loss curve ──
if train_losses:
    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.plot(train_steps, train_losses, linewidth=1.5, color="steelblue", label="Train loss")
    ax.set_xlabel("Optimizer step", fontsize=12)
    ax.set_ylabel("Loss", fontsize=12)
    ax.set_title("NoseKnows — Training Loss Curve", fontsize=14, fontweight="bold")
    ax.legend(fontsize=11)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    plt.tight_layout()
    p = PLOTS_DIR / "train_loss_curve.png"
    plt.savefig(p, dpi=150)
    plt.show()
    print(f"Saved: {p}")

# ── Plot 2: validation loss per epoch ──
if eval_losses:
    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.plot(
        eval_epochs, eval_losses,
        marker="o", linewidth=2, markersize=8,
        color="coral", label="Validation loss",
    )
    for x, y in zip(eval_epochs, eval_losses):
        ax.annotate(
            f"{y:.4f}", (x, y),
            textcoords="offset points", xytext=(0, 10),
            ha="center", fontsize=9,
        )
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Loss", fontsize=12)
    ax.set_title("NoseKnows — Validation Loss per Epoch", fontsize=14, fontweight="bold")
    ax.legend(fontsize=11)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    plt.tight_layout()
    p = PLOTS_DIR / "val_loss_per_epoch.png"
    plt.savefig(p, dpi=150)
    plt.show()
    print(f"Saved: {p}")

# ── Plot 3: train vs validation comparison ──
if train_losses and eval_losses:
    steps_per_epoch_actual = max(train_steps) / NUM_EPOCHS if NUM_EPOCHS > 0 else 1
    eval_steps_approx = [e * steps_per_epoch_actual for e in eval_epochs]

    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.plot(train_steps, train_losses, linewidth=1.2, color="steelblue",
            alpha=0.8, label="Train loss")
    ax.plot(eval_steps_approx, eval_losses, marker="o", linewidth=2,
            markersize=8, color="coral", label="Validation loss")
    ax.set_xlabel("Optimizer step", fontsize=12)
    ax.set_ylabel("Loss", fontsize=12)
    ax.set_title("NoseKnows — Train vs Validation Loss", fontsize=14, fontweight="bold")
    ax.legend(fontsize=11)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    plt.tight_layout()
    p = PLOTS_DIR / "train_vs_val_loss.png"
    plt.savefig(p, dpi=150)
    plt.show()
    print(f"Saved: {p}")

print("\nAll plots saved to", PLOTS_DIR)

In [ ]:
# ── Cell 13: inference sanity check ──────────────────────────────────────
#
# System prompt merged into user turn at inference time,
# matching exactly the format used during training.

NOSKNOWS_SYSTEM_PROMPT = (
    "You are NoseKnows, a fragrance consultant who knows perfumery inside out. "
    "When someone describes what they are after, whether a mood, an occasion, "
    "or notes they love or cannot stand, you recommend real perfumes by name and "
    "brand and explain exactly why they fit, grounding your answer in the actual "
    "notes and accords. Warm, confident, specific. Never vague, never a catalogue. "
    "3 to 5 sentences."
)

TEST_QUERIES = [
    "I want something warm and cozy for autumn evenings, not too sweet.",
    "Looking for a fresh citrus scent for the office, nothing too loud.",
    "I love oud and rose but can't stand anything synthetic smelling.",
]


def run_inference(query: str) -> str:
    """Text-only inference with the fine-tuned model."""
    # Merge system prompt into user turn — matches training format exactly.
    merged = f"{NOSKNOWS_SYSTEM_PROMPT}\n\n{query}"
    messages = [
        {"role": "user", "content": merged},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    if hasattr(input_ids, "input_ids"):
        input_ids = input_ids.input_ids
    elif isinstance(input_ids, dict):
        input_ids = input_ids["input_ids"]

    input_ids = input_ids.to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


print("=" * 60)
print("INFERENCE SANITY CHECK")
print("=" * 60)
for query in TEST_QUERIES:
    response = run_inference(query)
    print(f"\nUSER    : {query}")
    print(f"NOSKNOWS: {response}")
    print("-" * 60)

In [ ]:
# ── Cell 14: final summary ────────────────────────────────────────────────

print("\n" + "=" * 62)
print("FINE-TUNING COMPLETE")
print("=" * 62)
print(f"  Model              : {MODEL_ID}")
print(f"  Loading class      : AutoModelForCausalLM")
print(f"  LoRA layers        : last {N_LORA_LAYERS} transformer layers")
print(f"  LoRA r / alpha     : {LORA_R} / {LORA_ALPHA}")
print(f"  Train examples     : {len(train_dataset):,}")
print(f"  Val examples       : {len(val_dataset):,}")
print(f"  Epochs completed   : {NUM_EPOCHS}")
print(f"  Total steps        : {train_result.global_step}")
print(f"  Final train loss   : {train_result.training_loss:.4f}")
if loss_history_cb.eval_losses:
    best_val = min(l for _, l in loss_history_cb.eval_losses)
    print(f"  Best val loss      : {best_val:.4f}")
print(
    f"  Runtime            : "
    f"{train_result.metrics.get('train_runtime', 0) / 3600:.2f}h"
)
print("=" * 62)
print(f"\nFinal adapter     → {FINAL_ADAPTER}")
print(f"Epoch checkpoints → {CHECKPOINT_DIR}/epoch_N/")
print(f"Plots             → {PLOTS_DIR}")
print(f"Report            → {REPORT_PATH}")
print("\nDownload final_adapter/ from the Output tab.")